In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import glob
import cv2

train_imgs_path = '../input/petfinder-pawpularity-score/train'
train_imgs = np.array([cv2.resize(cv2.imread(file),(224,224)) for file in glob.glob(train_imgs_path+'/*.jpg')])
train_csv = pd.read_csv('../input/petfinder-pawpularity-score/train.csv')
train_data = np.array(train_csv.drop(columns=['Id','Pawpularity']))
train_label = np.array(train_csv['Pawpularity'])


2026-01-07 04:50:23.560924: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767761423.574423      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767761423.578569      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 04:50:23.595585: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# define two sets of inputs
inputA = keras.Input(shape=(12))
inputB = keras.Input(shape=(224,224,3))
# the first branch operates on the first input
x = layers.Dense(12, activation="softmax")(inputA)
x = keras.Model(inputs=inputA, outputs=x)
# the second branch opreates on the second input
y = layers.Conv2D(96, (11,11),padding='same', activation='relu', strides=4)(inputB)
y = layers.MaxPooling2D(pool_size=(3,3), strides=2)(y)
y = layers.Conv2D(256, (5,5),padding='same', activation='relu', strides=2)(y)
y = layers.MaxPooling2D(pool_size=(3,3), strides=2)(y)
y = layers.Conv2D(384, (3,3),padding='same', activation='relu')(y)
y = layers.Conv2D(384, (3,3),padding='same', activation='relu')(y)
y = layers.Conv2D(256, (3,3),padding='same', activation='relu')(y)
y = layers.MaxPooling2D(pool_size=(3,3), strides=2)(y)
y = layers.Flatten()(y)
y = layers.Dense(12, activation="softmax")(y)
y = keras.Model(inputs=inputB, outputs=y)
# combine the output of the two branches
combined = layers.Concatenate()([x.output, y.output])
# apply a FC layer and then a regression prediction on the
# combined outputs
z = layers.Dense(24, activation="relu")(combined)
z = layers.Dense(1, activation="linear")(z)

model = keras.Model([x.input, y.input], z)

keras.utils.plot_model(model, show_shapes=True)

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
)
 
model.fit([train_data[:7930],train_imgs[:7930]],train_label[:7930], epochs=25)
results = model.evaluate([train_data[7930:],train_imgs[7930:]],train_label[7930:])
print("test loss, test acc:", results)


ValueError: Cannot convert '12' to a shape.

In [3]:
test_csv = pd.read_csv("../input/petfinder-pawpularity-score/test.csv") 
test_imgs_path = '../input/petfinder-pawpularity-score/test'
test_imgs = np.array([cv2.resize(cv2.imread(file),(224,224)) for file in glob.glob(test_imgs_path+'/*.jpg')])
    
prediction = model.predict([test_csv.drop(columns="Id"),test_imgs])
count = 0
data = []
for i in test_csv.itertuples():
  data.append([i[1],prediction[count][0]])
  count+=1
prediction_df = pd.DataFrame(data,columns=['Id','Pawpularity'])
prediction_df.to_csv('submission.csv',header=True,index=None)
prediction_df


NameError: name 'model' is not defined